# 02 — Tool Signal Analysis

**Purpose**: Analyze the quality, diversity, and drift of tool signals.

All data from SQLite only.

**Sections**:
1. Tool output distributions (histograms)
2. Mean and std per tool
3. Tool-to-tool correlation matrix (heatmap)
4. Rolling correlation with market price
5. Tool drift over time (rolling mean)
6. Identify near-zero variance tools

In [ ]:
import sys
from pathlib import Path
REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from config import SQLITE_DB_FILE
from prediction_agent.storage.sqlite_store import SQLiteStore

store = SQLiteStore()

# Load all tool_outputs joined with runs
df = pd.DataFrame(store.query("""
    SELECT
        t.run_id,
        r.timestamp,
        r.score AS final_score,
        t.tool_id,
        t.output_mean,
        t.weight,
        t.contribution,
        r.outcome
    FROM tool_outputs t
    JOIN runs r ON t.run_id = r.run_id
    ORDER BY r.timestamp
"""))

if df.empty:
    print('⚠ No tool_outputs data found in SQLite. Run the live loop first.')
else:
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce')
    print(f'Loaded {len(df)} tool_output rows across {df["tool_id"].nunique()} tools')
    print(f'Date range: {df["timestamp"].min()} → {df["timestamp"].max()}')
    print(df.groupby('tool_id')['output_mean'].describe().round(4))

## 1. Tool Output Distributions

In [ ]:
if not df.empty:
    tools = df['tool_id'].unique()
    n_tools = len(tools)
    ncols = min(3, n_tools)
    nrows = (n_tools + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
    axes = np.array(axes).flatten() if n_tools > 1 else [axes]

    for i, tool in enumerate(sorted(tools)):
        vals = df[df['tool_id'] == tool]['output_mean'].dropna()
        axes[i].hist(vals, bins=30, color='steelblue', alpha=0.75, edgecolor='white')
        axes[i].set_title(tool, fontsize=10)
        axes[i].set_xlabel('output_mean')
        axes[i].set_ylabel('Count')
        axes[i].axvline(vals.mean(), color='red', linestyle='--', linewidth=1.5, label=f'μ={vals.mean():.3f}')
        axes[i].legend(fontsize=8)

    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Tool Output Distributions', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

## 2. Mean and Std Per Tool

In [ ]:
if not df.empty:
    stats = df.groupby('tool_id')['output_mean'].agg(['mean','std','min','max','count']).round(4)
    stats.columns = ['Mean', 'Std', 'Min', 'Max', 'N']
    print('Tool Signal Statistics:')
    print(stats.to_string())

    fig, ax = plt.subplots(figsize=(10, 4))
    x = range(len(stats))
    ax.bar(x, stats['Mean'], yerr=stats['Std'], color='steelblue', alpha=0.8,
           capsize=5, error_kw={'linewidth':1.5})
    ax.set_xticks(list(x))
    ax.set_xticklabels(stats.index, rotation=30, ha='right')
    ax.set_title('Tool Output Mean ± Std', fontsize=13)
    ax.set_ylabel('output_mean')
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='0.5 reference')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 3. Tool-to-Tool Correlation Matrix

In [ ]:
if not df.empty:
    # Pivot: rows = run_id, cols = tool_id, values = output_mean
    pivot = df.pivot_table(index='run_id', columns='tool_id', values='output_mean', aggfunc='mean')

    if pivot.shape[1] > 1:
        corr = pivot.corr()

        fig, ax = plt.subplots(figsize=(max(6, len(corr)*1.2), max(5, len(corr)*1.0)))
        cmap = plt.cm.RdYlGn
        im = ax.imshow(corr, cmap=cmap, vmin=-1, vmax=1, aspect='auto')
        plt.colorbar(im, ax=ax, label='Pearson r')

        ax.set_xticks(range(len(corr.columns)))
        ax.set_yticks(range(len(corr.index)))
        ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=9)
        ax.set_yticklabels(corr.index, fontsize=9)

        for i in range(len(corr)):
            for j in range(len(corr.columns)):
                val = corr.iloc[i, j]
                ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8,
                        color='black' if abs(val) < 0.8 else 'white')

        ax.set_title('Tool-to-Tool Pearson Correlation Matrix', fontsize=13)
        plt.tight_layout()
        plt.show()

        # Flag high correlations (potential redundancy)
        high_corr = []
        for i in range(len(corr)):
            for j in range(i+1, len(corr.columns)):
                r = corr.iloc[i, j]
                if abs(r) > 0.85:
                    high_corr.append((corr.index[i], corr.columns[j], round(r, 4)))

        if high_corr:
            print('⚠ Highly correlated tool pairs (|r| > 0.85):')
            for a, b, r in high_corr:
                print(f'  {a} ↔ {b}:  r={r}')
        else:
            print('✓ No tool pairs with |r| > 0.85.')
    else:
        print('Need ≥2 tools for correlation matrix.')

## 4. Rolling Correlation with Market Price

In [ ]:
if not df.empty:
    runs_df = pd.DataFrame(store.query(
        'SELECT run_id, timestamp, score FROM runs ORDER BY timestamp'
    ))
    runs_df['timestamp'] = pd.to_datetime(runs_df['timestamp'], utc=True, errors='coerce')

    if len(runs_df) > 20:
        merged = df.merge(runs_df[['run_id', 'score']], on='run_id')
        tools = merged['tool_id'].unique()

        fig, axes = plt.subplots(len(tools), 1, figsize=(12, 3*len(tools)), sharex=True)
        if len(tools) == 1:
            axes = [axes]

        WINDOW = min(50, len(merged) // 4)
        for ax, tool in zip(axes, sorted(tools)):
            tool_df = merged[merged['tool_id'] == tool].sort_values('timestamp')
            roll_corr = tool_df['output_mean'].rolling(WINDOW).corr(tool_df['score'])
            ax.plot(tool_df['timestamp'], roll_corr, linewidth=1.5, label=tool)
            ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
            ax.set_ylabel('Rolling r', fontsize=9)
            ax.set_title(f'{tool} — rolling corr with final_score (window={WINDOW})', fontsize=10)
            ax.set_ylim(-1, 1)

        plt.suptitle('Rolling Correlation: Tool Output vs Final Score', fontsize=13)
        plt.tight_layout()
        plt.show()
    else:
        print(f'Need >20 runs for rolling correlation (have {len(runs_df)}).')

## 5. Tool Drift Over Time

In [ ]:
if not df.empty and len(df) > 20:
    tools = sorted(df['tool_id'].unique())
    WINDOW = min(30, len(df) // (len(tools) * 2))

    fig, ax = plt.subplots(figsize=(14, 5))
    for tool in tools:
        tool_df = df[df['tool_id'] == tool].sort_values('timestamp').reset_index(drop=True)
        rolling_mean = tool_df['output_mean'].rolling(WINDOW, min_periods=5).mean()
        ax.plot(tool_df['timestamp'], rolling_mean, linewidth=1.8, label=tool)

    ax.axhline(0.5, color='black', linestyle=':', alpha=0.3)
    ax.set_title(f'Tool Output Rolling Mean (window={WINDOW})', fontsize=13)
    ax.set_xlabel('Time (UTC)')
    ax.set_ylabel('Rolling Mean output')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print('Not enough data for drift plot.')

## 6. Identify Near-Zero Variance Tools

In [ ]:
VARIANCE_THRESHOLD = 0.001

if not df.empty:
    stds = df.groupby('tool_id')['output_mean'].std()
    low_var = stds[stds < VARIANCE_THRESHOLD]

    if len(low_var) > 0:
        print(f'⚠ Tools with near-zero variance (std < {VARIANCE_THRESHOLD}):')
        for tool, std in low_var.items():
            print(f'  {tool}: std={std:.6f}')
        print('  These tools produce constant output and contribute no signal.')
    else:
        print(f'✓ All tools have sufficient variance (std ≥ {VARIANCE_THRESHOLD}).')

    print('\nFull std table:')
    print(stds.sort_values().to_string())